# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reetuparabat/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [15]:
"""
1) Two findings from FlyRank's "The State of AI-Driven SEO, March 2026" paper,
and the methodology question I'd ask about each -- constructive, the way I'd
want my own ML-08 work reviewed.

--------------------------------------------------------------------------
FINDING 1 -- "What Predicts Health?" (ML Appendix, p.27)

The paper trains a Random Forest to predict health_score and reports feature
importance: Average Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8%.
The paper's own Methodology section discloses the health_score formula:
Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)
-- a composite built directly from the same four inputs the model is scored on.
To their credit, the paper already flags this in one sentence: importance is
"descriptive rather than causal" because the target is "partly constructed
from some of these inputs."

My methodology question: how much of that 43%+32%=75% combined importance is
the model rediscovering the scoring formula's own weights, versus finding
something the formula doesn't already encode? Position and impressions are
literally 60 of the composite's 100 points by construction, so a large chunk
of "importance" here may be arithmetic, not discovery. A concrete fix I'd
ask for: report importance computed AFTER removing the score's own direct
components from the feature set (i.e., what predicts health_score using only
content age, word count, days visible, AI sessions -- the four features that
scored near 0% here) -- or better, predict something the formula doesn't
already contain, like future traffic growth. Right now the chart can't tell
a reader how much of the 75% is circular.

--------------------------------------------------------------------------
FINDING 2 -- "What Predicts Growth?" (ML Appendix, p.29)

A Logistic Regression reports 71% holdout accuracy classifying growing vs.
declining pages, with content age as the strongest negative signal and
"days visible and recent impressions" among the strongest positive signals.
The paper's growth/decline label is defined earlier (Definitions, p.6) as a
30-day-vs-previous-30-day impression comparison. The Methodology page
discloses an 80/20 split for this model but does not say whether that split
was grouped by brand or done at the row level -- and this portfolio spans
just 57 brands across 61.8K content pieces, meaning a random row split would
almost certainly put many pages from the same brand in both the training set
and the holdout set.

My methodology question, in two parts: first, is 71% actually accuracy on a
label that might already be, say, 55-60% one class? The paper doesn't state
the class balance next to the number, and per the base-rate rule, 71% next
to an undisclosed base rate can't be judged as skill or coincidence. Second,
if "recent impressions" in the coefficient chart means impressions from the
same last-30-day window the label is computed from, that's a feature drawn
from inside the label's own defining window, not just correlated with it --
worth a one-line clarification on whether "recent" here means the label's
own last-30d window or a distinct, earlier one. Both questions are answerable
from data the paper's authors already have; I'm not asking for new data
collection, just two disclosures next to an existing chart.
"""


'\n1) Two findings from FlyRank\'s "The State of AI-Driven SEO, March 2026" paper,\nand the methodology question I\'d ask about each -- constructive, the way I\'d\nwant my own ML-08 work reviewed.\n\n--------------------------------------------------------------------------\nFINDING 1 -- "What Predicts Health?" (ML Appendix, p.27)\n\nThe paper trains a Random Forest to predict health_score and reports feature\nimportance: Average Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8%.\nThe paper\'s own Methodology section discloses the health_score formula:\nImpressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)\n-- a composite built directly from the same four inputs the model is scored on.\nTo their credit, the paper already flags this in one sentence: importance is\n"descriptive rather than causal" because the target is "partly constructed\nfrom some of these inputs."\n\nMy methodology question: how much of that 43%+32%=75% combined importance is\nthe model

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [16]:
!git clone https://github.com/reetuparabat/flyrank-ml-internship.git
!ls flyrank-ml-internship/data/raw

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
content_refresh_anonymized.csv


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv('/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
TARGET = 'is_declining_label'

# same exclusions as ML-08, re-declared here so this notebook stands alone
LEAKY = ['trend_direction', 'trend_pct',
         'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
         'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
IDS = ['content_id', 'client_id']
DROP_LOW_VALUE = ['provider_used', 'model_used']

feature_cols = [c for c in df.columns if c not in LEAKY + IDS + DROP_LOW_VALUE + [TARGET]]
numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
categorical_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]
SYSTEMATIC_MISSING = ['word_count', 'char_count', 'search_volume', 'competition', 'cpc', 'main_intent']

def build_pipeline(num_cols, cat_cols):
    preprocess = ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
    ])
    return Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))])

def add_missing_flags(train_df, test_df, cols):
    train_df, test_df = train_df.copy(), test_df.copy()
    flags = []
    for col in cols:
        if col in train_df.columns:
            train_df[f'{col}_was_missing'] = train_df[col].isna().astype(int)
            test_df[f'{col}_was_missing'] = test_df[col].isna().astype(int)
            flags.append(f'{col}_was_missing')
    return train_df, test_df, flags

def precision_at_k(frame, score_col, k):
    return frame.sort_values(score_col, ascending=False).head(k)[TARGET].mean()


In [18]:
# --- BEFORE: random row-level split (the naive, dishonest default) ---
train_r, test_r = train_test_split(df, test_size=0.25, random_state=RANDOM_SEED, stratify=df[TARGET])
train_r, test_r, flags_r = add_missing_flags(train_r, test_r, SYSTEMATIC_MISSING)
num_final_r = numeric_cols + flags_r

overlap_random = set(train_r['client_id']) & set(test_r['client_id'])
print(f'Client overlap in RANDOM split: {len(overlap_random)} of {df["client_id"].nunique()} total clients')

pipe_r = build_pipeline(num_final_r, categorical_cols)
pipe_r.fit(train_r[num_final_r + categorical_cols], train_r[TARGET])
test_r = test_r.copy()
test_r['score'] = pipe_r.predict_proba(test_r[num_final_r + categorical_cols])[:, 1]
auc_r = roc_auc_score(test_r[TARGET], test_r['score'])
p20_r = precision_at_k(test_r, 'score', 20)
p50_r = precision_at_k(test_r, 'score', 50)
print(f'RANDOM split  -> AUC: {auc_r:.3f} | precision@20: {p20_r:.3f} | precision@50: {p50_r:.3f}')


Client overlap in RANDOM split: 31 of 32 total clients
RANDOM split  -> AUC: 0.695 | precision@20: 1.000 | precision@50: 0.900


In [19]:
# --- AFTER: grouped-by-client split (the honest design, matching ML-08) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_g, test_g = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
train_g, test_g, flags_g = add_missing_flags(train_g, test_g, SYSTEMATIC_MISSING)
num_final_g = numeric_cols + flags_g

overlap_grouped = set(train_g['client_id']) & set(test_g['client_id'])
print(f'Client overlap in GROUPED split: {len(overlap_grouped)} of {df["client_id"].nunique()} total clients')

pipe_g = build_pipeline(num_final_g, categorical_cols)
pipe_g.fit(train_g[num_final_g + categorical_cols], train_g[TARGET])
test_g['score'] = pipe_g.predict_proba(test_g[num_final_g + categorical_cols])[:, 1]
auc_g = roc_auc_score(test_g[TARGET], test_g['score'])
p20_g = precision_at_k(test_g, 'score', 20)
p50_g = precision_at_k(test_g, 'score', 50)
print(f'GROUPED split -> AUC: {auc_g:.3f} | precision@20: {p20_g:.3f} | precision@50: {p50_g:.3f}')

print()
print('=== BEFORE/AFTER GAP ===')
print(f'AUC gap (random - grouped):          {auc_r - auc_g:+.3f}')
print(f'precision@20 gap (random - grouped): {p20_r - p20_g:+.3f}')
print(f'precision@50 gap (random - grouped): {p50_r - p50_g:+.3f}')


Client overlap in GROUPED split: 0 of 32 total clients
GROUPED split -> AUC: 0.582 | precision@20: 0.850 | precision@50: 0.680

=== BEFORE/AFTER GAP ===
AUC gap (random - grouped):          +0.113
precision@20 gap (random - grouped): +0.150
precision@50 gap (random - grouped): +0.220


In [20]:
"""
2) Before/after: what the gap shows

The random split put 31 of my 32 clients on both sides of the train/test
boundary -- effectively the whole dataset. Under that split, the model
looked much stronger than it is: AUC 0.695 and a perfect precision@20 of
1.000. Under the grouped split (0 client overlap, confirmed), the same
model, same features, same seed, drops to AUC 0.582 and precision@20 of
0.850.

That gap (+0.113 AUC, +0.150 precision@20) is memorization, not skill --
the random-split model was partly learning "which brand is this" rather
than "is this page declining," because it had already seen other pages
from the same brand during training. This is the exact ML-08 number
(0.582 AUC) reproduced here under the same grouped design, which is a
useful consistency check: my Week-5 result wasn't a fluke of that
particular run.
"""


'\n2) Before/after: what the gap shows\n\nThe random split put 31 of my 32 clients on both sides of the train/test\nboundary -- effectively the whole dataset. Under that split, the model\nlooked much stronger than it is: AUC 0.695 and a perfect precision@20 of\n1.000. Under the grouped split (0 client overlap, confirmed), the same\nmodel, same features, same seed, drops to AUC 0.582 and precision@20 of\n0.850.\n\nThat gap (+0.113 AUC, +0.150 precision@20) is memorization, not skill --\nthe random-split model was partly learning "which brand is this" rather\nthan "is this page declining," because it had already seen other pages\nfrom the same brand during training. This is the exact ML-08 number\n(0.582 AUC) reproduced here under the same grouped design, which is a\nuseful consistency check: my Week-5 result wasn\'t a fluke of that\nparticular run.\n'

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [21]:
# The attack checklist, run against my actual ML-08 / Section 2 feature set
print('[x] Timeline drawn: all features are pre-decision 90-day aggregates or')
print('    content/keyword attributes; label is a 30d-vs-prev-30d comparison.')
print()
print('[x] No label-derived columns in features -- excluded:', LEAKY)
print()
print('[x] No product-flags as features -- ML-07 baseline_score is compared')
print('    AGAINST the model, never fed into it as an input.')
print()
print(f'[x] Split grouped by client_id -- confirmed 0 overlap above.')
print()
print(f'[x] Base rate printed: {test_g[TARGET].mean():.3f} (test-set base rate, same population')
print(f'    precision@20 was measured on) vs. grouped-test precision@20 of {p20_g:.3f}')
print()
print('[ ] Metrics recomputed out-of-fold: NOT done -- this is a single grouped')
print('    holdout, not k-fold. Naming this as a real limitation, not hiding it.')


[x] Timeline drawn: all features are pre-decision 90-day aggregates or
    content/keyword attributes; label is a 30d-vs-prev-30d comparison.

[x] No label-derived columns in features -- excluded: ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

[x] No product-flags as features -- ML-07 baseline_score is compared
    AGAINST the model, never fed into it as an input.

[x] Split grouped by client_id -- confirmed 0 overlap above.

[x] Base rate printed: 0.517 (test-set base rate, same population
    precision@20 was measured on) vs. grouped-test precision@20 of 0.850

[ ] Metrics recomputed out-of-fold: NOT done -- this is a single grouped
    holdout, not k-fold. Naming this as a real limitation, not hiding it.


In [22]:
# Deliberate leakage-injection test, per the skill's own verification method:
# add a known-leaky column back and confirm the score actually jumps.
# If it didn't jump, the test harness itself would be broken.
leaky_feature = 'impressions_last_30d'
num_final_leak = num_final_g + [leaky_feature]
pipe_leak = build_pipeline(num_final_leak, categorical_cols)
pipe_leak.fit(train_g[num_final_leak + categorical_cols], train_g[TARGET])
test_g['score_leaky'] = pipe_leak.predict_proba(test_g[num_final_leak + categorical_cols])[:, 1]
auc_leak = roc_auc_score(test_g[TARGET], test_g['score_leaky'])

print(f"Grouped split, HONEST features:                    AUC = {auc_g:.3f}")
print(f"Grouped split, WITH '{leaky_feature}' added back:  AUC = {auc_leak:.3f}")
print(f"Jump from adding the leaky column: {auc_leak - auc_g:+.3f}")


Grouped split, HONEST features:                    AUC = 0.582
Grouped split, WITH 'impressions_last_30d' added back:  AUC = 0.723
Jump from adding the leaky column: +0.141


In [23]:
"""
3) What the leakage audit found

Top feature importance in ML-08 (permutation importance, days_with_impressions
leading, no single feature above ~0.03 AUC-drop) already looked like a healthy
spread rather than one column doing all the work -- the classic symptom of
label-derived leakage (one feature towering over the rest, score near 1.0)
was absent, which was reassuring but not proof by itself.

This notebook adds the actual proof: deliberately putting impressions_last_30d
back into the honest, grouped-split feature set jumps AUC from 0.582 to 0.723
(+0.141) -- a real, meaningful jump. That confirms two things at once: my test
harness correctly detects leakage when it's present (so the ML-08 "clean"
result wasn't clean because the test was broken), and excluding that column
in ML-08 was the right call, not overcaution -- it really was carrying
label information, exactly as the empirical check in ML-08 showed
(r=0.9999999984 between impressions_last_30d/prev_30d and trend_pct).

Real limitation I'm not hiding: this is one grouped train/test split, not
k-fold cross-validation. With only 32 clients, a single split's exact numbers
could move on a different seed -- I'd want 5-fold grouped CV before treating
0.582 as a precise number rather than a directional one.
"""


'\n3) What the leakage audit found\n\nTop feature importance in ML-08 (permutation importance, days_with_impressions\nleading, no single feature above ~0.03 AUC-drop) already looked like a healthy\nspread rather than one column doing all the work -- the classic symptom of\nlabel-derived leakage (one feature towering over the rest, score near 1.0)\nwas absent, which was reassuring but not proof by itself.\n\nThis notebook adds the actual proof: deliberately putting impressions_last_30d\nback into the honest, grouped-split feature set jumps AUC from 0.582 to 0.723\n(+0.141) -- a real, meaningful jump. That confirms two things at once: my test\nharness correctly detects leakage when it\'s present (so the ML-08 "clean"\nresult wasn\'t clean because the test was broken), and excluding that column\nin ML-08 was the right call, not overcaution -- it really was carrying\nlabel information, exactly as the empirical check in ML-08 showed\n(r=0.9999999984 between impressions_last_30d/prev_30d and

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [24]:
"""
4) My boldest ML-08 sentence, rewritten in safe language

ORIGINAL (from ML-08's interpretation section):
"Logistic Regression is the clear winner at every K... a real, large lift...
Random Forest is weaker than Logistic Regression at every K... despite being
the more complex model."

What's actually too strong here: "clear winner" and "real, large lift" state
a conclusion as settled fact from a single train/test split on 32 clients --
exactly the kind of claim this notebook's own Section 2 just showed can move
with the split. "Despite being the more complex model" is fine (it's a
description of what happened), but the framing around it oversells certainty
this one run doesn't support.

REWRITTEN, safe language:
"On this single grouped split, Logistic Regression OUTPERFORMED Random
Forest at every K tested (precision@20/50/100) -- a DIRECTIONAL result
consistent with the idea that added model complexity isn't earning its
keep on this feature set, though a 32-client single split is not enough
to call this a settled comparison. This is DECISION-SUPPORT for preferring
the simpler model for now, not proof it will hold under 5-fold grouped CV
or on next month's data."

Also rewriting my Section 2 claim above the same way: "the random-split
model was partly learning which brand is this" is a causal claim I can't
fully prove from one comparison -- the safer version is "the random-split
model's higher score is DIRECTIONALLY CONSISTENT with client memorization,
given 31-of-32 client overlap, though I have not isolated client identity
as the sole cause of the gap."
"""


'\n4) My boldest ML-08 sentence, rewritten in safe language\n\nORIGINAL (from ML-08\'s interpretation section):\n"Logistic Regression is the clear winner at every K... a real, large lift...\nRandom Forest is weaker than Logistic Regression at every K... despite being\nthe more complex model."\n\nWhat\'s actually too strong here: "clear winner" and "real, large lift" state\na conclusion as settled fact from a single train/test split on 32 clients --\nexactly the kind of claim this notebook\'s own Section 2 just showed can move\nwith the split. "Despite being the more complex model" is fine (it\'s a\ndescription of what happened), but the framing around it oversells certainty\nthis one run doesn\'t support.\n\nREWRITTEN, safe language:\n"On this single grouped split, Logistic Regression OUTPERFORMED Random\nForest at every K tested (precision@20/50/100) -- a DIRECTIONAL result\nconsistent with the idea that added model complexity isn\'t earning its\nkeep on this feature set, though a 32-

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.